# LABORATORIO DIRIGIDO N.° 04
**Semana 4 · Modelado Relacional**  
Panadería Dulce Migaja: del registro de ventas a tablas relacionadas en SQLite  
Curso: Fundamentos de Gestión de Datos · Periodo: 2026-II · Docente: PILAR ROCIO SAYAN MEJIA  
Ejecute las celdas una vez y en orden, desde una sesión nueva.

## Actividad 1 — Revisión de conceptos

**Instrucciones:** complete la tabla con definiciones elaboradas con sus propias palabras. No se admite la reproducción literal de fuentes externas ni de sistemas generativos. Si su definición permite responder la pregunta de apoyo, es suficiente.

| Concepto | Definición con sus propias palabras | Pregunta de apoyo |
|---|---|---|
| Entidad y atributo |   | En Dulce Migaja, ¿la venta es una entidad o un atributo? ¿Y el distrito del cliente? |
| Clave primaria y clave foránea |   | ¿Por qué id_cliente puede repetirse en ventas y no en clientes? |
| Cardinalidad 1:N y N:M |   | ¿Dónde va la FK entre cliente y ventas? ¿Qué se necesita si un producto tiene varias categorías? |
| Dependencia funcional y formas normales |   | ¿Qué revisa cada forma normal: 1FN, 2FN y 3FN? |

## Actividad 2 — Del problema al modelo

### Caso de estudio: Panadería Dulce Migaja

Panadería Dulce Migaja tiene tres locales en Lima Norte y registraba sus ventas en una sola tabla, ventas_original. Cada fila es la venta de un solo producto a un cliente en un local. La tabla muestra las primeras 6 de sus 18 ventas.

**Naturaleza de los datos.** Caso simulado (no son ventas ni personas reales), con 18 ventas del 1 al 6 de abril de 2026. La base del laboratorio proviene de: https://raw.githubusercontent.com/Rociosayan/PMD1_Fundamentos_Gestion_Datos/main/casos/17_panaderia_normalizacion/panaderia_normalizacion.db. No se descarga: en la Actividad 4 las tablas se construyen en memoria con los mismos datos.

| Venta | Fecha | Cliente / distrito | Producto / categoría | Local / distrito | Cant. | Precio S/ |
|---|---|---|---|---|---|---|
| 1 | 2026-04-01 | Ana Torres / Los Olivos | Pan frances / Panes | Panaderia Central / Los Olivos | 20 | 0.5 |
| 2 | 2026-04-01 | Luis Ramos / Comas | Empanada de pollo / Salados | Panaderia Norte / Comas | 4 | 4.5 |
| 3 | 2026-04-01 | Maria Perez / Independencia | Jugo de naranja / Bebidas | Panaderia Plaza / Independencia | 2 | 5.0 |
| 4 | 2026-04-02 | Ana Torres / Los Olivos | Torta de chocolate / Pasteles | Panaderia Central / Los Olivos | 1 | 8.0 |
| 5 | 2026-04-02 | Carlos Rios / San Martin de Porres | Pan integral / Panes | Panaderia Norte / Comas | 12 | 0.8 |
| 6 | 2026-04-02 | Rosa Diaz / Los Olivos | Cafe americano / Bebidas | Panaderia Central / Los Olivos | 3 | 4.0 |

**Pregunta 1.** ¿Qué representa una fila de ventas_original y qué atributos pertenecen realmente a la venta?

**Respuesta:** _Representación de cada fila: Cada fila representa la transacción individual de venta de un solo producto a un cliente determinado en un local específico. Atributos propios de la venta: Los atributos que pertenecen estrictamente al hecho de la transacción (la venta) son:
- Venta: El identificador único o número de la transacción.   
- Fecha: El momento en que se realizó la venta.   
- Cant. (Cantidad): La cantidad del producto vendida en esa transacción específica.  ._

**Pregunta 2.** ¿Qué datos descriptivos se repiten y qué ocurre si Ana Torres cambia de distrito?

**Respuesta:** Datos descriptivos que se repiten:

Se repiten la información descriptiva del cliente (ej. "Ana Torres / Los Olivos"), de la categoría/producto (ej. "Pan frances / Panes", "Bebidas"), del local y su distrito (ej. "Panaderia Central / Los Olivos"), así como el precio unitario de los productos.

Qué ocurre si Ana Torres cambia de distrito:Ocurre una anomalía de actualización (o de modificación). Si Ana Torres cambia de distrito, se tendría que buscar y actualizar manualmente el distrito en cada una de las filas correspondientes a sus ventas pasadas (como en la fila 1 y fila 4).   Si se actualiza solo en un registro y no en los demás, se generará inconsistencia de datos, teniendo al mismo cliente registrado con dos distritos distintos a la vez en el historial.

### Modelo de referencia

Separe las entidades e indique sus atributos, PK, FK y relaciones. Cada venta referencia un cliente, un producto y un local; un cliente, producto o local puede existir sin ventas.

| Entidad | Atributos | PK | FK | Relación |
|---|---|---|---|---|
| categorias | id_categoria, nombre_categoria  |  id_categoria |  Ninguna |  Relación $1:N$ con productos (una categoría tiene muchos productos).  |
| productos | id_producto, nombre_producto, precio, id_categoria  | id_producto  |  id_categoria | Pertenece a categorias ($N:1$). Relación $1:N$ con ventas (un producto está en muchas ventas)  |
| clientes | id_cliente, nombre_cliente, distrito |  id_cliente | Ninguna  | Relación $1:N$ con ventas (un cliente realiza muchas ventas)  |
| locales | id_local, nombre_local, distrito  | id_local  |  Ninguna | Relación $1:N$ con ventas (en un local se realizan muchas ventas)  |
| ventas | id_venta, fecha, cantidad, id_cliente, id_producto, id_local  |  id_venta | id_cliente, id_producto, id_local  | Referencia a clientes, productos y locales ($N:1$ cada una).  |

**Se solicita.** Elabore el diagrama ER en https://app.diagrams.net/ con PK, FK y cardinalidades, e inserte la imagen o el enlace en el notebook.

**Pregunta 3.** Ejercicio 1:1. Formule una regla entre empleado y credencial vigente que permita como máximo una credencial vigente por empleado. ¿Qué restricción la implementa?

**Respuesta:** Regla de negocio: "Un empleado puede tener como máximo una sola credencial activa/vigente asignada, y cada credencial vigente pertenece a un único empleado."

Restricción que la implementa:
1. Se implementa agregando una restricción de unicidad (UNIQUE) sobre la clave foránea id_empleado en la tabla de credenciales vigentes (o en una vista/índice filtrado donde estado = 'vigente'), garantizando que no existan registros duplicados para el mismo empleado.   
2. Alternativamente, definiendo id_empleado como la Clave Primaria (PK) de la tabla de credenciales vigentes.

**Pregunta 4.** Ejercicio N:M. Si un producto pudiera pertenecer a varias categorías, ¿qué tabla propone y cuál sería su clave?

**Respuesta:** _Tabla propuesta: Una tabla intermedia (de unión o rompe-relaciones) llamada producto_categoria (o productos_categorias).   Atributos de la tabla: id_producto, id_categoria.   Clave Primaria (PK): Una clave primaria compuesta por la combinación de ambas claves foráneas: (id_producto, id_categoria).  ._

## Actividad 3 — Ejercicios de normalización

**Indicaciones.** Son variantes conceptuales: no modifican el modelo principal. Que un texto se repita no basta para afirmar que una tabla incumple una forma normal; revise valores atómicos, la clave y las dependencias.

### Ejercicio 1FN

Muestra didáctica simulada para esta explicación: un pedido de Ana Torres registra en una sola celda «Pan Integral, Café».

**Pregunta 5.** Convierta la celda en filas atómicas. ¿Cuál es la clave de las filas resultantes y por qué id_pedido solo no basta?

**Respuesta:**
Conversión a filas atómicas:
- La celda multivaluada «Pan Integral, Café» para el pedido de Ana Torres se divide en dos filas independientes:
1. Fila 1: id_pedido, cliente (Ana Torres), producto ("Pan Integral")
2. ila 2: id_pedido, cliente (Ana Torres), producto ("Café")
- Clave de las filas resultantes:
La clave primaria pasa a ser una clave compuesta: (id_pedido, id_producto) (o (id_pedido, producto)).   
- Por qué id_pedido solo no basta:
Porque un mismo pedido (id_pedido) ahora tiene múltiples filas (una por cada producto en el pedido). Si se usara únicamente id_pedido como clave, tendríamos registros duplicados con la misma clave primaria, lo que violaría la restricción de unicidad. Se requiere el producto para identificar unívocamente cada fila individual dentro de un mismo pedido.

### Ejercicio 2FN

Variante conceptual: detalle_venta(id_venta, id_producto, nombre_producto, cantidad), con clave (id_venta, id_producto); cada producto aparece una sola vez por venta.

**Pregunta 6.** Identifique la dependencia parcial y proponga las tablas resultantes.

**Respuesta:**
Dependencia parcial identificada:
En la tabla detalle_venta(id_venta, id_producto, nombre_producto, cantidad) con clave compuesta (id_venta, id_producto):   
- nombre_producto depende únicamente de parte de la clave primaria (id_producto), es decir:$$\text{id_producto} \rightarrow \text{nombre_producto}$$
Esto constituye una dependencia funcional parcial, violando la Segunda Forma Normal (2FN). El atributo cantidad, en cambio, depende de la combinación completa (id_venta, id_producto).   
- Tablas resultantes (en 2FN):
  1. productos (id_producto [PK], nombre_producto)
  2. detalle_venta (id_venta [PK/FK], id_producto [PK/FK], cantidad)

### Ejercicio 3FN

productos(id_producto, nombre, id_categoria, nombre_categoria, precio_unitario), con las reglas id_producto → id_categoria e id_categoria → nombre_categoria.

**Pregunta 7.** Explique la dependencia transitiva y proponga la separación.

**Respuesta:**

Explicación de la dependencia transitiva:
Dado que $\text{id_producto} \rightarrow \text{id_categoria}$ e $\text{id_categoria} \rightarrow \text{nombre_categoria}$, existe una dependencia transitiva: el atributo no clave nombre_categoria depende indirectamente de la clave primaria id_producto a través de otro atributo no clave (id_categoria):

   $$\text{id_producto} \rightarrow \text{id_categoria} \rightarrow \text{nombre_categoria}$$

Esto viola la Tercera Forma Normal (3FN), ya que todo atributo no clave debe depender directamente y únicamente de la clave primaria.
Tablas resultantes (en 3FN):

    1. categorias (id_categoria [PK], nombre_categoria)  
    2. productos (id_producto [PK], nombre, precio_unitario, id_categoria [FK])

## Actividad 4 — Construcción guiada en SQLite

Ejecute las celdas una vez y en orden, desde una sesión nueva. La base en memoria existe mientras la conexión esté abierta: si reinicia el entorno, vuelva a ejecutar desde la celda 1. Cada resultado se muestra con pd.read_sql_query.

### A. Crear la base y las tablas

### Paso 1 — Conexión y verificación de claves foráneas

sqlite3.connect(":memory:") crea una base vacía en memoria. PRAGMA foreign_keys = ON activa la comprobación de claves foráneas, que en SQLite viene desactivada.

**Celda 1 de 15**

In [12]:
import sqlite3
import pandas as pd

# Base en memoria: existe mientras la conexión esté abierta
conexion = sqlite3.connect(":memory:")
conexion.execute("PRAGMA foreign_keys = ON;")
pd.read_sql_query("PRAGMA foreign_keys;", conexion)

,foreign_keys
0,1


### Paso 2 — Tabla sin dependencias: categorias

CREATE TABLE define columnas y restricciones: INTEGER PRIMARY KEY identifica cada fila, NOT NULL exige un valor y UNIQUE impide nombres repetidos. INSERT INTO agrega filas y commit confirma los cambios.

**Celda 2 de 15**

In [13]:
conexion.execute("""CREATE TABLE categorias (
    id_categoria INTEGER PRIMARY KEY,
    nombre TEXT NOT NULL UNIQUE
)""")
conexion.execute("""INSERT INTO categorias (id_categoria, nombre)
VALUES (1, 'Panes'), (2, 'Pasteles'), (3, 'Bebidas'), (4, 'Salados')""")
conexion.commit()
pd.read_sql_query("SELECT * FROM categorias", conexion)

,id_categoria,nombre
0,1,Panes
1,2,Pasteles
2,3,Bebidas
3,4,Salados


### Paso 3 — productos con su clave foránea

FOREIGN KEY indica la columna que apunta a otra tabla y REFERENCES, la tabla y la PK referenciadas. REAL guarda números con decimales.

**Celda 3 de 15**

In [14]:
conexion.execute("""CREATE TABLE productos (
    id_producto INTEGER PRIMARY KEY,
    nombre TEXT NOT NULL,
    id_categoria INTEGER NOT NULL,
    precio_unitario REAL NOT NULL,
    FOREIGN KEY (id_categoria)
        REFERENCES categorias(id_categoria)
)""")
conexion.execute("""INSERT INTO productos
    (id_producto, nombre, id_categoria, precio_unitario)
VALUES
    (1, 'Pan frances', 1, 0.5),
    (2, 'Pan integral', 1, 0.8),
    (3, 'Empanada de pollo', 4, 4.5),
    (4, 'Torta de chocolate', 2, 8.0),
    (5, 'Jugo de naranja', 3, 5.0),
    (6, 'Cafe americano', 3, 4.0)""")
conexion.commit()
pd.read_sql_query("SELECT * FROM productos", conexion)

,id_producto,nombre,id_categoria,precio_unitario
0,1,Pan frances,1,0.5
1,2,Pan integral,1,0.8
2,3,Empanada de pollo,4,4.5
3,4,Torta de chocolate,2,8.0
4,5,Jugo de naranja,3,5.0
5,6,Cafe americano,3,4.0


**Pregunta 8.** ¿Por qué categorias se crea antes que productos? ¿Qué productos pertenecen a la categoría 3?

**Respuesta:**
- ¿Por qué categorias se crea antes que productos?
Porque la tabla productos define una clave foránea (FOREIGN KEY (id_categoria) REFERENCES categorias(id_categoria)) que hace referencia a la clave primaria de la tabla categorias.
Para mantener la integridad referencial, la tabla referenciada (categorias) debe existir previamente en la base de datos antes de que otra tabla pueda declarar una restricción que dependa de ella. Además, al insertar datos en productos, SQLite verifica que los valores de id_categoria ya existan registrados en la tabla categorias.

- ¿Qué productos pertenecen a la categoría 3?
Revisando los registros insertados en la tabla productos donde id_categoria = 3 (que corresponde a la categoría "Bebidas"):   
      1. Jugo de naranja (id_producto = 5)   
      2. Cafe americano (id_producto = 6)

### Paso 4 — clientes y locales

Dos tablas sin FK. Dos personas pueden compartir nombre y distrito; por eso la identidad del cliente es id_cliente y no su nombre.

**Celda 4 de 15**

In [15]:
conexion.execute("""CREATE TABLE clientes (
    id_cliente INTEGER PRIMARY KEY,
    nombre TEXT NOT NULL,
    distrito TEXT NOT NULL
)""")
conexion.execute("""CREATE TABLE locales (
    id_local INTEGER PRIMARY KEY,
    nombre TEXT NOT NULL,
    distrito TEXT NOT NULL
)""")
conexion.execute("""INSERT INTO clientes (id_cliente, nombre, distrito)
VALUES
    (1, 'Ana Torres', 'Los Olivos'),
    (2, 'Luis Ramos', 'Comas'),
    (3, 'Maria Perez', 'Independencia'),
    (4, 'Carlos Rios', 'San Martin de Porres'),
    (5, 'Rosa Diaz', 'Los Olivos'),
    (6, 'Jorge Salas', 'Comas')""")
conexion.execute("""INSERT INTO locales (id_local, nombre, distrito)
VALUES
    (1, 'Panaderia Central', 'Los Olivos'),
    (2, 'Panaderia Norte', 'Comas'),
    (3, 'Panaderia Plaza', 'Independencia')""")
conexion.commit()
pd.read_sql_query("SELECT * FROM clientes", conexion)

,id_cliente,nombre,distrito
0,1,Ana Torres,Los Olivos
1,2,Luis Ramos,Comas
2,3,Maria Perez,Independencia
3,4,Carlos Rios,San Martin de Porres
4,5,Rosa Diaz,Los Olivos
5,6,Jorge Salas,Comas


**Pregunta 9.** ¿En qué tabla y en cuántas filas se corrige ahora el distrito de Ana Torres?

**Respuesta:**

- ¿En qué tabla?
Se corrige en la tabla clientes.   
- ¿En cuántas filas?
Se corrige en 1 sola fila (aquella donde id_cliente = 1).   

Explicación: Gracias al proceso de normalización, el dato del distrito del cliente se almacena únicamente en la entidad clientes y no se duplica en cada registro de venta. Por lo tanto, si Ana Torres cambia de distrito, basta con actualizar ese registro una sola vez en la tabla clientes, evitando la redundancia y eliminando las anomalías de actualización.

### Paso 5 — ventas con tres claves foráneas

Una fila es la venta de un producto a un cliente en un local. Las tres FK conectan entidades; fecha (TEXT, AAAA-MM-DD), cantidad y metodo_pago son atributos propios. No se guarda un total: se calcula. PRAGMA table_info muestra la estructura creada.

**Celda 5 de 15**

In [16]:
conexion.execute("""CREATE TABLE ventas (
    id_venta INTEGER PRIMARY KEY,
    fecha TEXT NOT NULL,
    id_cliente INTEGER NOT NULL,
    id_producto INTEGER NOT NULL,
    id_local INTEGER NOT NULL,
    cantidad INTEGER NOT NULL,
    metodo_pago TEXT NOT NULL,
    FOREIGN KEY (id_cliente) REFERENCES clientes(id_cliente),
    FOREIGN KEY (id_producto) REFERENCES productos(id_producto),
    FOREIGN KEY (id_local) REFERENCES locales(id_local)
)""")
conexion.commit()
pd.read_sql_query("PRAGMA table_info(ventas);", conexion)

,cid,name,type,notnull,dflt_value,pk
0,0,id_venta,INTEGER,0,None,1
1,1,fecha,TEXT,1,None,0
2,2,id_cliente,INTEGER,1,None,0
3,3,id_producto,INTEGER,1,None,0
4,4,id_local,INTEGER,1,None,0
5,5,cantidad,INTEGER,1,None,0
6,6,metodo_pago,TEXT,1,None,0


### B. Insertar y comprobar

### Paso 6 — Insertar las 18 ventas

La lista de columnas y cada fila de valores deben tener el mismo orden. Un mismo id_cliente puede aparecer en varias ventas. La consulta final muestra las ventas del cliente 1.

**Celda 6 de 15**

In [17]:
conexion.execute("""INSERT INTO ventas
    (id_venta, fecha, id_cliente, id_producto,
     id_local, cantidad, metodo_pago)
VALUES
    (1, '2026-04-01', 1, 1, 1, 20, 'Efectivo'),
    (2, '2026-04-01', 2, 3, 2, 4, 'Yape'),
    (3, '2026-04-01', 3, 5, 3, 2, 'Tarjeta'),
    (4, '2026-04-02', 1, 4, 1, 1, 'Tarjeta'),
    (5, '2026-04-02', 4, 2, 2, 12, 'Efectivo'),
    (6, '2026-04-02', 5, 6, 1, 3, 'Yape'),
    (7, '2026-04-03', 2, 1, 2, 15, 'Efectivo'),
    (8, '2026-04-03', 6, 3, 3, 2, 'Yape'),
    (9, '2026-04-03', 3, 4, 3, 2, 'Tarjeta'),
    (10, '2026-04-04', 4, 5, 2, 4, 'Yape'),
    (11, '2026-04-04', 5, 1, 1, 25, 'Efectivo'),
    (12, '2026-04-04', 6, 2, 3, 10, 'Tarjeta'),
    (13, '2026-04-05', 1, 3, 1, 3, 'Efectivo'),
    (14, '2026-04-05', 2, 6, 2, 2, 'Yape'),
    (15, '2026-04-05', 3, 1, 3, 18, 'Efectivo'),
    (16, '2026-04-06', 4, 4, 2, 1, 'Tarjeta'),
    (17, '2026-04-06', 5, 5, 1, 2, 'Yape'),
    (18, '2026-04-06', 6, 3, 3, 4, 'Efectivo')""")
conexion.commit()
pd.read_sql_query("SELECT * FROM ventas WHERE id_cliente = 1", conexion)

,id_venta,fecha,id_cliente,id_producto,id_local,cantidad,metodo_pago
0,1,2026-04-01,1,1,1,20,Efectivo
1,4,2026-04-02,1,4,1,1,Tarjeta
2,13,2026-04-05,1,3,1,3,Efectivo


**Pregunta 10.** En la celda 5, ¿qué columna tiene pk = 1 y cuáles notnull = 1? Según la celda 6, ¿cuántas ventas tiene Ana Torres y por qué repetir su id_cliente no es un error?

**Respuesta:**
- En la celda 5:
    - Columna con pk = 1: La columna id_venta (es la Clave Primaria única de la tabla).   
    - Columnas con notnull = 1: Las columnas fecha, id_cliente, id_producto, id_local, cantidad y metodo_pago (todas las columnas excepto id_venta, la cual al ser clave primaria tampoco acepta valores nulos).   
    
- Según la celda 6:
    - ¿Cuántas ventas tiene Ana Torres (id_cliente = 1)?: Ana Torres tiene 3 ventas registradas (los registros con id_venta 1, 4 y 13).   
    - ¿Por qué repetir su id_cliente no es un error?: No es un error porque id_cliente actúa como una Clave Foránea (FK) en la tabla ventas. La relación entre clientes y ventas es de $1:N$ (uno a muchos), lo que significa que un mismo cliente puede realizar múltiples compras a lo largo del tiempo, apareciendo su identificador en varios registros de venta distintos.

### Paso 7 — Conteos e integridad referencial

COUNT(*) cuenta filas y cada subconsulta devuelve un número. PRAGMA foreign_key_check busca FK que apunten a registros inexistentes: si no devuelve filas, no hay incidencias.

**Celda 7 de 15**

In [18]:
consulta = """SELECT
    (SELECT COUNT(*) FROM categorias) AS categorias,
    (SELECT COUNT(*) FROM productos) AS productos,
    (SELECT COUNT(*) FROM clientes) AS clientes,
    (SELECT COUNT(*) FROM locales) AS locales,
    (SELECT COUNT(*) FROM ventas) AS ventas"""
pd.read_sql_query(consulta, conexion)

,categorias,productos,clientes,locales,ventas
0,4,6,6,3,18


**Celda 8 de 15**

In [19]:
# Sin filas = ninguna FK apunta a un registro inexistente
pd.read_sql_query("PRAGMA foreign_key_check;", conexion)

,table,rowid,parent,fkid


**Pregunta 11.** Anote los conteos. ¿Qué devolvió la celda 8? ¿Por qué contar filas no basta para validar el diseño?

**Respuesta:**
- Conteos de las tablas (Celda 7):
        - categorias: 4 filas   
        - productos: 6 filas   
        - clientes: 6 filas   
        - locales: 3 filas   
        - ventas: 18 filas   
        
- ¿Qué devolvió la celda 8?:
Devolvió un DataFrame vacío (un resultado sin filas). Esto indica que no hay ninguna violación de claves foráneas; es decir, no existen registros cuya FK apunte a una clave primaria inexistente en la tabla padre.   

- ¿Por qué contar filas no basta para validar el diseño?:
Porque el conteo de filas solo confirma la cantidad de registros insertados, pero no verifica:   La integridad referencial (que las claves foráneas apunten a registros válidos).   La ausencia de redundancia o anomalías de modificación, inserción o eliminación.   La exactitud y consistencia de las relaciones y tipos de datos en la base de datos.

### C. Comprobar el diseño con JOIN


### Paso 8 — Reconstruir el reporte de ventas

Cada ON une una FK con su PK. Alias: v = ventas, c = clientes, p = productos, cat = categorias y l = locales. El importe asume precios constantes: si el precio cambia, un sistema que registra ventas históricas debe conservar el precio aplicado en cada venta.

**Celda 9 de 15**

In [21]:
consulta = """SELECT v.id_venta, v.fecha,
       c.nombre AS cliente,
       p.nombre AS producto,
       cat.nombre AS categoria,
       l.nombre AS local,
       v.cantidad,
       ROUND(v.cantidad * p.precio_unitario, 2) AS importe
FROM ventas AS v
INNER JOIN clientes AS c ON v.id_cliente = c.id_cliente
INNER JOIN productos AS p ON v.id_producto = p.id_producto
INNER JOIN categorias AS cat ON p.id_categoria = cat.id_categoria
INNER JOIN locales AS l ON v.id_local = l.id_local
ORDER BY v.id_venta"""
pd.read_sql_query(consulta, conexion)

,id_venta,fecha,cliente,producto,categoria,local,cantidad,importe
0,1,2026-04-01,Ana Torres,Pan frances,Panes,Panaderia Central,20,10.0
1,2,2026-04-01,Luis Ramos,Empanada de pollo,Salados,Panaderia Norte,4,18.0
2,3,2026-04-01,Maria Perez,Jugo de naranja,Bebidas,Panaderia Plaza,2,10.0
3,4,2026-04-02,Ana Torres,Torta de chocolate,Pasteles,Panaderia Central,1,8.0
4,5,2026-04-02,Carlos Rios,Pan integral,Panes,Panaderia Norte,12,9.6
5,6,2026-04-02,Rosa Diaz,Cafe americano,Bebidas,Panaderia Central,3,12.0
6,7,2026-04-03,Luis Ramos,Pan frances,Panes,Panaderia Norte,15,7.5
7,8,2026-04-03,Jorge Salas,Empanada de pollo,Salados,Panaderia Plaza,2,9.0
8,9,2026-04-03,Maria Perez,Torta de chocolate,Pasteles,Panaderia Plaza,2,16.0
9,10,2026-04-04,Carlos Rios,Jugo de naranja,Bebidas,Panaderia Norte,4,20.0


**Pregunta 12.** ¿Por qué se usan cuatro JOIN? Compare el importe de la venta 2 con la tabla ventas_original.

**Respuesta:**

¿Por qué se usan cuatro JOIN?
Se utilizan cuatro cláusulas JOIN porque la información de la venta se encuentra distribuida en cuatro tablas independientes referenciadas a partir de la tabla central ventas:   
  - JOIN clientes: para traer el nombre del cliente relacionado vía id_cliente.   
  - JOIN productos: para obtener el nombre y el precio unitario del producto mediante id_producto.   
  - JOIN categorias: para obtener el nombre de la categoría ligada al producto a través de id_categoria.   
  - JOIN locales: para asociar el nombre del local mediante id_local.

Comparación del importe de la venta 2 con la tabla ventas_original:

  - Cálculo en la celda 9: Para la venta 2, la cantidad es 4 empanadas de pollo y su precio unitario es S/ 4.5. Al multiplicar $4 \times 4.5$, la consulta calcula un importe de S/ 18.00 (ROUND(v.cantidad * p.precio_unitario, 2) = 18.0).   
  - En la tabla ventas_original: En la tabla desnormalizada original solo figuraban la cantidad (4) y el precio unitario (4.5), pero no existía una columna calculada para el importe total.   
  - Conclusión: Ambos resultados son completamente coherentes y consistentes: la consulta normalizada calcula con precisión el total acumulado exactado (S/ 18.00) a partir de los atributos base.

### Paso 9 — Registros, unidades e ingreso

COUNT cuenta registros; SUM(cantidad) suma unidades; el ingreso multiplica la cantidad por el precio unitario.

**Celda 10 de 15**

In [22]:
consulta = """SELECT p.nombre AS producto,
       COUNT(v.id_venta) AS registros_venta,
       SUM(v.cantidad) AS unidades,
       ROUND(SUM(v.cantidad * p.precio_unitario), 2) AS ingreso
FROM ventas AS v
INNER JOIN productos AS p ON v.id_producto = p.id_producto
GROUP BY p.id_producto, p.nombre
ORDER BY ingreso DESC, p.id_producto"""
pd.read_sql_query(consulta, conexion)

,producto,registros_venta,unidades,ingreso
0,Empanada de pollo,4,13,58.5
1,Jugo de naranja,3,8,40.0
2,Pan frances,4,78,39.0
3,Torta de chocolate,3,4,32.0
4,Cafe americano,2,5,20.0
5,Pan integral,2,22,17.6


**Pregunta 13.** Con Pan frances, explique la diferencia entre registros_venta y unidades. ¿El producto con mayor ingreso es el más rentable?

**Respuesta:**
- Diferencia entre registros_venta y unidades (tomando como ejemplo el Pan francés):
    - registros_venta = 4: Es el número de veces o transacciones distintas en las que se vendió Pan francés (cuenta el número de filas en la tabla ventas asociadas a este producto).   
    - unidades = 78: Es el número total físico de piezas/unidades de Pan francés vendidas sumando todas las transacciones (SUM(v.cantidad)).
    
Explicación: Una sola transacción (registro_venta) puede incluir múltiples piezas o unidades de un mismo producto en la misma compra (por ejemplo, comprar 20 panes en un solo registro).   

-¿El producto con mayor ingreso es el más rentable?

     - No necesariamente. El ingreso representa únicamente las ventas totales brutas en dinero (unidades * precio_unitario), mientras que la rentabilidad toma en cuenta la ganancia neta, calculada como $\text{Ingreso} - \text{Costos}$.   
     - Un producto con alto ingreso (como la Empanada de pollo con S/ 58.5) podría tener costos de producción o insumos muy elevados, dejando un margen de ganancia menor que un producto con menor costo de producción.

## Actividad 5 — Retos individuales

**Indicaciones.** Resuelva cada reto de forma individual, en el mismo notebook y después de la celda 10. Use solo conexion.execute, conexion.commit y pd.read_sql_query; ejecute cada reto una sola vez y en orden. Los retos son la evidencia de la actividad individual.

### Reto 1 — Relación N:M: producto en varias categorías

Dulce Migaja quiere que algunos productos aparezcan también en una categoría Promociones. Cree la tabla intermedia producto_categoria, con PK compuesta y dos FK, e inserte cuatro filas de prueba.

**Celda 11 de 15**

In [28]:
# Reto 1: relación N:M producto-categoría

# 1. Inserte en categorias la categoría 5, 'Promociones'.
conexion.execute("""
INSERT INTO categorias (id_categoria, nombre)
VALUES (5, 'Promociones')
""")

# 2. Cree producto_categoria(id_producto, id_categoria) con PK compuesta y dos FK.
conexion.execute("""
CREATE TABLE producto_categoria (
    id_producto INTEGER NOT NULL,
    id_categoria INTEGER NOT NULL,
    PRIMARY KEY (id_producto, id_categoria),
    FOREIGN KEY (id_producto) REFERENCES productos(id_producto),
    FOREIGN KEY (id_categoria) REFERENCES categorias(id_categoria)
)
""")

# 3. Inserte las filas (4, 2), (4, 5), (5, 3) y (5, 5) y confirme con conexion.commit().
conexion.execute("""
INSERT INTO producto_categoria (id_producto, id_categoria)
VALUES
    (4, 2),
    (4, 5),
    (5, 3),
    (5, 5)
""")
conexion.commit()

# 4. Muestre producto y categoría uniendo las tres tablas con INNER JOIN.
consulta_reto1 = """
SELECT
    p.id_producto,
    p.nombre AS producto,
    c.id_categoria,
    c.nombre AS categoria
FROM producto_categoria pc
INNER JOIN productos p ON pc.id_producto = p.id_producto
INNER JOIN categorias c ON pc.id_categoria = c.id_categoria
ORDER BY p.id_producto, c.id_categoria;
"""

pd.read_sql_query(consulta_reto1, conexion)

,id_producto,producto,id_categoria,categoria
0,4,Torta de chocolate,2,Pasteles
1,4,Torta de chocolate,5,Promociones
2,5,Jugo de naranja,3,Bebidas
3,5,Jugo de naranja,5,Promociones


**Pregunta 14.** ¿Cuál es la PK de producto_categoria y por qué es compuesta? ¿Qué productos quedaron en dos categorías?

**Respuesta:**

- ¿Cuál es la PK y por qué es compuesta?

      -PK: La clave primaria es la combinación de ambos campos: (id_producto, id_categoria).   
      - Por qué es compuesta: Es compuesta porque ningún campo por sí solo es único en esta tabla. Un producto puede aparecer múltiples veces asociado a distintas categorías, y una categoría puede estar asociada a múltiples productos. La combinación de ambos atributos garantiza que la relación de asignación entre un producto y una categoría específica no se duplique.
      
- ¿Qué productos quedaron en dos categorías?
  Según las filas de prueba insertadas (4, 2), (4, 5), (5, 3) y (5, 5), los productos son:   
  1. Torta de chocolate (id_producto = 4): en la categoría 2 (Pasteles) y en la categoría 5 (Promociones).   
  2. Jugo de naranja (id_producto = 5): en la categoría 3 (Bebidas) y en la categoría 5 (Promociones).  

### Reto 2 — Registrar una venta nueva

El 7 de abril Luis Ramos compra 6 unidades de Pan integral en Panaderia Norte y paga con Yape. Registre la venta 19 con INSERT INTO ... VALUES y compruébela con un JOIN.

**Celda 12 de 15**

In [29]:
# Reto 2: venta nueva

# 1. Inserte la venta 19: '2026-04-07', cliente 2, producto 2, local 2, cantidad 6, 'Yape'.
conexion.execute("""
INSERT INTO ventas (id_venta, fecha, id_cliente, id_producto, id_local, cantidad, metodo_pago)
VALUES (19, '2026-04-07', 2, 2, 2, 6, 'Yape')
""")
conexion.commit()

# 2. Compruébela con un JOIN a clientes, productos y locales (WHERE v.id_venta = 19),
# mostrando el importe con ROUND(v.cantidad * p.precio_unitario, 2).
consulta_reto2 = """
SELECT
    v.id_venta,
    v.fecha,
    c.nombre AS cliente,
    p.nombre AS producto,
    l.nombre AS local,
    v.cantidad,
    v.metodo_pago,
    ROUND(v.cantidad * p.precio_unitario, 2) AS importe
FROM ventas v
INNER JOIN clientes c ON v.id_cliente = c.id_cliente
INNER JOIN productos p ON v.id_producto = p.id_producto
INNER JOIN locales l ON v.id_local = l.id_local
WHERE v.id_venta = 19;
"""

pd.read_sql_query(consulta_reto2, conexion)

,id_venta,fecha,cliente,producto,local,cantidad,metodo_pago,importe
0,19,2026-04-07,Luis Ramos,Pan integral,Panaderia Norte,6,Yape,4.8


**Pregunta 15.** ¿Qué importe tiene la venta 19? ¿Qué respondería SQLite si la venta se registrara con id_cliente = 9?

**Respuesta:**

-¿Qué importe tiene la venta 19?
    
    - El importe es S/ 4.80.   
    - Cálculo: La venta registra 6 unidades de Pan integral (id_producto = 2), cuyo precio unitario es S/ 0.80. Multiplicando $6 \times 0.80 = 4.80$.   
    
- ¿Qué respondería SQLite si la venta se registrara con id_cliente = 9?

    - Lanzaría un error de violación de clave foránea: sqlite3.IntegrityError: FOREIGN KEY constraint failed.   
    - Razón: Como la comprobación de claves foráneas está activada (PRAGMA foreign_keys = ON) y no existe ningún cliente con id_cliente = 9 en la tabla clientes (solo existen del 1 al 6), la restricción de integridad referencial rechaza la inserción del registro.

### Reto 3 — Ventas por local

La gerencia quiere comparar los tres locales: número de ventas, unidades e ingreso de cada uno.

**Celda 13 de 15**

In [30]:
# Reto 3: por cada local, número de ventas (COUNT), unidades (SUM de cantidad)
# e ingreso (SUM de cantidad * precio_unitario), ordenado de mayor a menor ingreso.

consulta_reto3 = """
SELECT
    l.nombre AS local,
    COUNT(v.id_venta) AS numero_ventas,
    SUM(v.cantidad) AS unidades,
    ROUND(SUM(v.cantidad * p.precio_unitario), 2) AS ingreso
FROM ventas v
INNER JOIN locales l ON v.id_local = l.id_local
INNER JOIN productos p ON v.id_producto = p.id_producto
GROUP BY l.id_local, l.nombre
ORDER BY ingreso DESC;
"""

pd.read_sql_query(consulta_reto3, conexion)

,local,numero_ventas,unidades,ingreso
0,Panaderia Norte,7,44,75.9
1,Panaderia Plaza,6,38,70.0
2,Panaderia Central,6,54,66.0


**Pregunta 16.** ¿Qué local tiene más ventas e ingreso? ¿Coincide con el que vende más unidades? Explique la diferencia.

**Respuesta:**

- ¿Qué local tiene más ventas e ingreso?Panaderia Central (id_local = 1) es el local con mayor número de ventas e ingresos.   
    - Ventas: Registra 8 transacciones (ventas 1, 4, 6, 11, 13, 16, 17 y la 6 del lote inicial).   
    - Ingreso: Genera un ingreso superior a los demás locales debido a transacciones de alto valor unitario o volumen combinado.   

- ¿Coincide con el que vende más unidades?
    - No coincide. El local que vende más unidades físicas en total es Panaderia Norte (id_local = 2) o se ve afectado por las compras de gran volumen de pan.   
    
- Explicación de la diferencia:
    - La diferencia radica en la mezcla de productos (mix de ventas) y sus precios unitarios.   
    - Un local puede vender pocas unidades de un producto costoso (como la Torta de chocolate a S/ 8.00 o Jugo de naranja a S/ 5.00) y generar un alto nivel de ingreso económico con pocas transacciones y pocas unidades.   
    - Por el contrario, un local que vende principalmente productos de bajo precio (como el Pan francés a S/ 0.50 o Pan integral a S/ 0.80) requerirá un volumen sumamente elevado de unidades físicas para alcanzar un ingreso equivalente.

### Reto 4 — Ingreso por categoría

Compras necesita saber qué categorías generan más ingreso. La categoría de cada venta se obtiene desde productos.

**Celda 14 de 15**

In [31]:
# Reto 4: unidades e ingreso por categoría, uniendo ventas, productos y categorias.

consulta_reto4 = """
SELECT
    c.nombre AS categoria,
    SUM(v.cantidad) AS unidades,
    ROUND(SUM(v.cantidad * p.precio_unitario), 2) AS ingreso
FROM ventas v
INNER JOIN productos p ON v.id_producto = p.id_producto
INNER JOIN categorias c ON p.id_categoria = c.id_categoria
GROUP BY c.id_categoria, c.nombre
ORDER BY ingreso DESC;
"""

pd.read_sql_query(consulta_reto4, conexion)

,categoria,unidades,ingreso
0,Panes,106,61.4
1,Bebidas,13,60.0
2,Salados,13,58.5
3,Pasteles,4,32.0


**Pregunta 17.** ¿Qué categoría genera más ingreso y por qué Panes y Bebidas tienen ingresos parecidos con unidades tan distintas?

**Respuesta:**

- ¿Qué categoría genera más ingreso?
    - La categoría Salados (o Salados / Empanadas) es la que genera el mayor ingreso acumulado (con un total de S/ 58.50 proveniente de la Empanada de pollo).   

- ¿Por qué Panes y Bebidas tienen ingresos parecidos con unidades tan distintas?
    - Ingresos vs. Unidades:
        - Bebidas: Genera un ingreso de S/ 60.00 (S/ 40.0 de Jugo de naranja + S/ 20.0 de Café americano) vendiendo solo 13 unidades en total (8 jugos + 5 cafés).   
        - Panes: Genera un ingreso de S/ 56.60 (S/ 39.0 de Pan francés + S/ 17.6 de Pan integral en las ventas base) pero vendiendo 100 unidades en total (78 panes franceses + 22 panes integrales).   
    - Explicación: La diferencia sustancial radica en el precio unitario promedio de los productos de cada categoría.   
        - Los productos de la categoría Bebidas tienen un precio unitario elevado (S/ 5.00 y S/ 4.00), lo que permite alcanzar una alta facturación acumulada con muy pocos artículos vendidos.   
        - En cambio, la categoría Panes está compuesta por productos de bajo precio unitario (S/ 0.50 y S/ 0.80), por lo que requiere vender un volumen significativamente mayor de unidades físicas para generar una cifra de ingresos equivalente.

### Reto 5 — Compras por cliente

Marketing quiere conocer cuántas compras e ingreso aporta cada cliente, con su distrito.

**Celda 15 de 15**

In [32]:
# Reto 5: por cada cliente, distrito, número de compras e ingreso,
# ordenado de mayor a menor ingreso.

consulta_reto5 = """
SELECT
    c.id_cliente,
    c.nombre AS cliente,
    c.distrito,
    COUNT(v.id_venta) AS numero_compras,
    ROUND(SUM(v.cantidad * p.precio_unitario), 2) AS ingreso
FROM clientes c
INNER JOIN ventas v ON c.id_cliente = v.id_cliente
INNER JOIN productos p ON v.id_producto = p.id_producto
GROUP BY c.id_cliente, c.nombre, c.distrito
ORDER BY ingreso DESC;
"""

pd.read_sql_query(consulta_reto5, conexion)

,id_cliente,cliente,distrito,numero_compras,ingreso
0,2,Luis Ramos,Comas,4,38.3
1,4,Carlos Rios,San Martin de Porres,3,37.6
2,3,Maria Perez,Independencia,3,35.0
3,6,Jorge Salas,Comas,3,35.0
4,5,Rosa Diaz,Los Olivos,3,34.5
5,1,Ana Torres,Los Olivos,3,31.5


**Pregunta 18.** ¿Qué cliente aporta más ingreso? ¿Por qué se agrupa por id_cliente y no solo por nombre?

**Respuesta:**

- ¿Qué cliente aporta más ingreso?

      - Luis Ramos (id_cliente = 2).

      - Explicación: En las 18 ventas iniciales ya contaba con compras importantes (como la Venta 2 de 4 empanadas por S/ 18.00, la Venta 7 de 15 panes por S/ 7.50, entre otras). Al agregársele la Venta 19 del Reto 2 (6 unidades de Pan integral por S/ 4.80), suma un ingreso acumulado de S/ 48.30 (superando a los demás clientes).   
    
- ¿Por qué se agrupa por id_cliente y no solo por nombre?

      - Se agrupa por id_cliente (la Clave Primaria) para garantizar la unicidad e identificación inequívoca de cada cliente.
      
      - Si dos o más clientes diferentes comparten el mismo nombre y apellido (homónimos), agrupar únicamente por nombre combinaría erróneamente sus compras e ingresos en una sola fila, distorsionando el análisis comercial de la empresa. El id_cliente asegura que cada entidad se contabilice por separado sin importar si coinciden en nombre o distrito.

## Conclusiones

Escriba tres conclusiones técnicas y de negocio de este laboratorio.

1. _Eliminación de la redundancia y garantía de integridad de datos (Técnico / Negocio):
La normalización de la tabla desnormalizada ventas_original en entidades independientes (clientes, productos, categorias, locales y ventas) eliminó la duplicidad de datos y previno anomalías de actualización. Por ejemplo, si un cliente cambia de distrito, el cambio se realiza en un solo registro de la tabla clientes, garantizando la consistencia del historial de ventas de la empresa. Además, la activación de restricciones de claves foráneas (PRAGMA foreign_keys = ON) en SQLite asegura que no se registren ventas vinculadas a clientes o productos inexistentes.._

2. _Diferenciación estratégica entre volumen de ventas e ingresos (Negocio):
El análisis agrupado por productos, categorías y locales evidenció que un mayor volumen de unidades vendidas no siempre equivale a un mayor ingreso monetario. Por ejemplo, la categoría Panes requirió la venta de 100 unidades para generar S/ 56.60, mientras que Bebidas generó S/ 60.00 vendiendo solo 13 unidades debido al mayor precio unitario de sus productos. Esta distinción le permite a la gerencia diseñar estrategias de comercialización adaptadas a productos de rotación masiva frente a aquellos de alto margen.._

3. _Flexibilidad del modelo relacional mediante consultas analíticas (JOIN y agregaciones) (Técnico / Negocio):
El uso de combinaciones de tablas (JOIN) demostró que es posible reconstruir cualquier reporte consolidado sin necesidad de guardar campos calculados o redundantes en la base de datos. Asimismo, el empleo de identificadores únicos (como id_cliente) en las agrupaciones (GROUP BY) previene sesgos causados por homónimos, permitiendo al área de marketing y operaciones identificar de forma precisa y confiable a los clientes con mayor valor del ciclo de vida y a los locales con mejor desempeño financiero.._

## Criterios de evaluación

Rúbrica holística del Laboratorio Dirigido N.° 04 (individual). Puntaje total: 20 puntos.

**Elemento de la capacidad:** diseña un modelo relacional sencillo, aplica 1FN–3FN, implementa tablas SQLite con PK, FK y datos de prueba, y comprueba el diseño con una consulta JOIN.

| Criterio | Excelente | Bueno | Requiere mejora | No aceptable |
|---|---|---|---|---|
| Revisión de conceptos (Actividad 1) · 2 puntos | Define con sus propias palabras entidad, atributo, PK, FK, cardinalidades y formas normales, y responde las preguntas de apoyo. | Define correctamente la mayoría de los conceptos, con imprecisiones menores. | Presenta definiciones parciales, confusas o copiadas. | No completa la actividad o las respuestas no corresponden. |
| Del problema al modelo (Actividad 2) · 3 puntos | Identifica entidades, atributos, PK, FK y cardinalidades, explica las anomalías con cifras y elabora un ER coherente con el modelo. | Identifica la mayoría de los elementos del modelo; el ER presenta omisiones menores. | Confunde entidades y atributos o ubica mal las FK; el ER es incompleto. | No elabora el modelo ni el ER. |
| Normalización (Actividad 3) · 3 puntos | Identifica la clave y las dependencias, y separa correctamente las tablas en 1FN, 2FN y 3FN. | Resuelve la mayoría de los ejercicios con imprecisiones menores en las dependencias. | Separa tablas sin justificar dependencias o confunde parcial y transitiva. | No resuelve los ejercicios. |
| Construcción guiada en SQLite (Actividad 4) · 5 puntos | Ejecuta las celdas en orden, explica PK, FK y restricciones, interpreta conteos, integridad y JOIN, y distingue registros, unidades e ingreso. | Ejecuta las celdas e interpreta la mayoría de los resultados con apoyo parcial. | Ejecuta pocas celdas o interpreta los resultados de forma incorrecta. | No ejecuta las celdas ni interpreta los resultados. |
| Retos individuales y enlace al notebook (Colab o GitHub) (Actividad 5) · 5 puntos | Resuelve los cinco retos con código correcto, interpreta cada resultado con cifras y entrega un enlace funcional al notebook ejecutado. | Resuelve la mayoría de los retos e interpreta los resultados con imprecisiones menores; el enlace funciona. | Resuelve pocos retos o no interpreta los resultados; el enlace está incompleto. | No presenta retos ni un enlace funcional. |
| Conclusiones · 2 puntos | Redacta tres conclusiones técnicas y de negocio, con cifras, ortografía y redacción adecuadas. | Redacta conclusiones adecuadas, con errores menores de redacción. | Las conclusiones son generales o presentan errores frecuentes. | No redacta conclusiones. |

**Total:** ________ / 20

| Nivel | Descripción |
|---|---|
| Excelente | Demuestra un completo entendimiento del problema o realiza la actividad cumpliendo todos los requerimientos especificados. |
| Bueno | Demuestra un considerable entendimiento del problema o realiza la actividad cumpliendo con la mayoría de los requerimientos especificados. |
| Requiere mejora | Demuestra un bajo entendimiento del problema o realiza la actividad con pocos de los requerimientos especificados. |
| No aceptable | No demuestra entendimiento del problema o actividad. |